In [0]:
%pip install mlflow

In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

import mlflow
import mlflow.sklearn

In [0]:
df = pd.read_csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv")

df.head()

In [0]:
df = df.dropna()

y = df['milliseconds']
X = df.drop(['milliseconds', 'time', 'duration'], axis=1)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 1. Build any model of your choice with tunable hyperparameters

In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

##2. Create an experiment setup where - for each run - you log: 

## the hyperparameters used in the model
## the model itself
## every possible metric from the model you chose
## at least two artifacts (plots, or csv files)

In [0]:
import mlflow
import mlflow.sklearn


mlflow.set_experiment("/Users/bt2683@columbia.edu/F1_pitstop_experiment")

with mlflow.start_run():

    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_squared_error, r2_score

    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    # log hyperparameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)

    # log metrics
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # log model
    mlflow.sklearn.log_model(model, "model")

    # artifact 1: prediction results
    import pandas as pd
    results = pd.DataFrame({
        "actual_milliseconds": y_test,
        "predicted_milliseconds": preds
    })
    results.to_csv("results.csv", index=False)
    mlflow.log_artifact("results.csv")

    # artifact 2: plot
    import matplotlib.pyplot as plt
    plt.scatter(y_test, preds)
    plt.xlabel("Actual milliseconds")
    plt.ylabel("Predicted milliseconds")
    plt.title("Prediction Plot")
    plt.savefig("plot.png")
    mlflow.log_artifact("plot.png")

## 3. Track your MLFlow experiment and run at least 10 experiments with different parameters each

In [0]:
for depth in [3, 5, 7, 10]:
    for trees in [50, 100, 200]:

        with mlflow.start_run():

            from sklearn.ensemble import RandomForestRegressor
            from sklearn.metrics import mean_squared_error, r2_score

            model = RandomForestRegressor(
                n_estimators=trees,
                max_depth=depth,
                random_state=42
            )

            model.fit(X_train, y_train)
            preds = model.predict(X_test)

            mse = mean_squared_error(y_test, preds)
            r2 = r2_score(y_test, preds)

            # log hyperparameters
            mlflow.log_param("n_estimators", trees)
            mlflow.log_param("max_depth", depth)

            # log metrics
            mlflow.log_metric("mse", mse)
            mlflow.log_metric("r2", r2)

            # log model
            mlflow.sklearn.log_model(model, "model")

## 4. Select your best model run and explain why

In [0]:
best_r2 = -1
best_params = None

for depth in [3, 5, 7, 10]:
    for trees in [50, 100, 200]:

        from sklearn.ensemble import RandomForestRegressor
        from sklearn.metrics import r2_score

        model = RandomForestRegressor(
            n_estimators=trees,
            max_depth=depth,
            random_state=42
        )

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        r2 = r2_score(y_test, preds)

        if r2 > best_r2:
            best_r2 = r2
            best_params = (depth, trees)

print("Best max_depth:", best_params[0])
print("Best n_estimators:", best_params[1])
print("Best R2:", best_r2)

The best model is selected based on the highest R² score among all experiments. The combination of max_depth and n_estimators that produces the highest R² is considered the best because it explains the largest proportion of variance in pit stop duration (milliseconds). This indicates that the model has better predictive performance compared to other parameter settings.